In [ ]:
from pathlib import Path
from typing import List, Tuple
from random import randint
from PIL import Image
import pandas as pd
import torch
import open_clip

In [ ]:
csv = (
    pd.read_csv(Path(".").joinpath("data").joinpath("annotations.csv"))
    .drop(["no", "anthropic"], axis=1)
    .assign(image=lambda x: x.image.astype(str))
)
csv

In [ ]:
columns = list(csv.columns[2:])
columns

In [ ]:
model_name = "ViT-B-16-SigLIP-512"
dataset_name = "webli"

model, _, preprocess = open_clip.create_model_and_transforms(
    model_name,
    pretrained=dataset_name,
)
model = model.to("cuda")
model.eval()
tokenizer = open_clip.get_tokenizer(model_name)

In [ ]:
row = csv.iloc[randint(0, len(csv))]
pil_img = Image.open(
    Path(".")
    .joinpath("data")
    .joinpath("images")
    .joinpath(row["image"])
    .with_suffix(".jpg")
)
pil_img

In [ ]:
image = preprocess(pil_img).unsqueeze(0)
text = tokenizer(columns)

with torch.no_grad(), torch.autocast("cuda"):
    image_features = model.encode_image(image.to("cuda"))
    text_features = model.encode_text(text.to("cuda"))
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

res_data = {"label": [], "prediction": [], "ground truth": []}

for class_item, class_likelihood in zip(columns, text_probs.detach().cpu().numpy()[0]):
    res_data["label"].append(class_item)
    res_data["prediction"].append(f"{class_likelihood:.2%}")
    res_data["ground truth"].append(row[class_item])

pd.DataFrame(data=res_data)